# Bag-of-Words SL

Sample script to exemplify **instantiating** and **supervised-learning** on the bag-of-words dataset. 

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sl import BagOfWordsSLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
    mse_expr
)

repo_root = get_repo_base()
device = torch.device("cuda:0")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/sl.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsSLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-example",
    corr=0.01,
    aux_words_ratio=0.5,
    train_epochs=2,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/config.py:131: UserWarning: /home/nlyu/Code/maxrl-statistics/artifacts/bow-sl-example/7-words_corr-0.01_len-128_pow-1.0_ar-0.5/config.json already exists with different config contents
  warnings.warn(


Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-sl-example/7-words_corr-0.01_len-128_pow-1.0_ar-0.5
Dataset corr target: 0.0100
Backbone lr: 1.230e-03
Head lr:     8.192e-03


/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/config.py:131: UserWarning: /home/nlyu/Code/maxrl-statistics/artifacts/bow-sl-example/7-words_corr-0.01_len-128_pow-1.0_ar-0.5/config.json already exists with different config contents
  warnings.warn(


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
state.run_training()

sl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

sl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

Training saves: 
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics. 
2. Validation parquet containing per-row ground-truth and target

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,7659.126465,19.513655,50331.882812,49984.0,7659.126465,-0.19978,5.014143,49984.0,1354.452759,-38.351517,49535.429688,49920.0,1354.452759,-0.352688,5.03975,49920.0
1,1732.102173,18.010138,50337.453125,49984.0,1732.102173,0.513272,5.013329,49984.0,1848.571777,51.964142,49535.429688,49920.0,1848.571777,0.342364,5.03975,49920.0


In [5]:
analysis = BagOfWordsAnalysisConfig.from_studies({"example": config.study_folder})
epoch_axis = pl.col("epoch").alias("epoch")

analysis.xy_plots([
    (epoch_axis, rsq_expr(split="train", y="ground_truth"), None),
    (epoch_axis, rsq_expr(split="val", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="train", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="val", y="ground_truth"), None),
])

In [6]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
-0.194336,-0.007019,0.073242
-0.1875,-0.006226,-1.984375
-0.197266,0.006226,-0.5625
-0.1875,0.007782,-2.15625
-0.197266,-0.004669,0.408203
